In [0]:
# Databricks notebook source
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

print("=" * 60)
print("FASE 3: FEATURE ENGINEERING")
print("=" * 60)

VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi"
DELTA_PATH = f"{VOLUME_PATH}/raw/delta"
PROCESSED_PATH = f"{VOLUME_PATH}/processed"

FILES = ["2015-01", "2016-01", "2016-02", "2016-03"]

# Carregar e combinar os 4 arquivos Delta
dfs = [spark.read.format("delta").load(f"{DELTA_PATH}/taxi_{periodo}") 
       for periodo in FILES]

df = dfs[0]
for d in dfs[1:]:
    df = df.unionByName(d)

print(f"\n📊 Linhas iniciais: {df.count():,}")
print(f"📋 Colunas iniciais: {len(df.columns)}")

In [0]:
# TAREFA 3.1: Limpeza baseada nos achados reais da Fase 2
print("\n" + "="*60)
print("TAREFA 3.1: Aplicando Regras de Limpeza")
print("="*60)

REGRAS_LIMPEZA = {
    "trip_distance_min": 0.1,
    "trip_distance_max": 200,      # milhas — baseado no p99.9 observado
    "fare_amount_min": 2.5,        # tarifa mínima NYC
    "fare_amount_max": 500,        # baseado no p99.9 observado
}

linhas_antes = df.count()

df_clean = df \
    .filter(col("trip_distance").between(
        REGRAS_LIMPEZA["trip_distance_min"], 
        REGRAS_LIMPEZA["trip_distance_max"])) \
    .filter(col("fare_amount").between(
        REGRAS_LIMPEZA["fare_amount_min"], 
        REGRAS_LIMPEZA["fare_amount_max"])) \
    .filter(col("tip_amount") >= 0) \
    .filter(col("total_amount") >= 0) \
    .filter(col("passenger_count").between(1, 8))

linhas_depois = df_clean.count()
removidas = linhas_antes - linhas_depois
pct_removido = (removidas / linhas_antes) * 100

print(f"📊 Linhas antes:   {linhas_antes:,}")
print(f"📊 Linhas depois:  {linhas_depois:,}")
print(f"🗑️  Removidas:      {removidas:,} ({pct_removido:.4f}%)")

In [0]:
# TAREFA 3.2: Conversão de Unidade
print("\n" + "="*60)
print("TAREFA 3.2: Conversão Milhas → KM")
print("="*60)

MILHAS_PARA_KM = 1.60934

df_featured = df_clean.withColumn(
    "trip_distance_km", 
    round(col("trip_distance") * MILHAS_PARA_KM, 2)
)

print("✅ Coluna 'trip_distance_km' criada (trip_distance original mantida em milhas)")
display(df_featured.select("trip_distance", "trip_distance_km").limit(5))

In [0]:
# TAREFA 3.3: Features Temporais
print("\n" + "="*60)
print("TAREFA 3.3: Features Temporais")
print("="*60)

df_featured = df_featured \
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_day_of_week", dayofweek(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_date", to_date(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_month", month(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_year", year(col("tpep_pickup_datetime"))) \
    .withColumn("is_weekend", 
        when((col("pickup_day_of_week") == 1) | (col("pickup_day_of_week") == 7), 1)
        .otherwise(0)) \
    .withColumn("is_rush_hour",
        when(((col("pickup_hour") >= 6) & (col("pickup_hour") <= 9)) |
             ((col("pickup_hour") >= 16) & (col("pickup_hour") <= 19)), 1)
        .otherwise(0)) \
    .withColumn("time_of_day",
        when(col("pickup_hour").between(6, 11), "manha")
        .when(col("pickup_hour").between(12, 16), "tarde")
        .when(col("pickup_hour").between(17, 21), "noite")
        .otherwise("madrugada"))

print("✅ Features criadas: pickup_hour, pickup_day_of_week, pickup_date,")
print("   pickup_month, pickup_year, is_weekend, is_rush_hour, time_of_day")

In [0]:
# TAREFA 3.4: Feature de Aeroporto (usando RatecodeID, não coordenadas)
print("\n" + "="*60)
print("TAREFA 3.4: Feature de Aeroporto via RatecodeID")
print("="*60)

df_featured = df_featured.withColumn(
    "is_airport_trip",
    when(col("RatecodeID").isin(2, 3), 1).otherwise(0)
).withColumn(
    "airport_type",
    when(col("RatecodeID") == 2, "JFK")
    .when(col("RatecodeID") == 3, "Newark")
    .otherwise("N/A")
)

print("✅ Features criadas: is_airport_trip, airport_type")
print("💡 Usamos RatecodeID (2=JFK, 3=Newark) em vez de calcular por coordenadas")

display(df_featured.groupBy("airport_type").count())

In [0]:
# TAREFA 3.5: Features de Velocidade
print("\n" + "="*60)
print("TAREFA 3.5: Features de Velocidade e Duração")
print("="*60)

df_featured = df_featured \
    .withColumn("trip_duration_minutes",
        (unix_timestamp(col("tpep_dropoff_datetime")) - 
         unix_timestamp(col("tpep_pickup_datetime"))) / 60) \
    .withColumn("duracao_valida",
        when(col("trip_duration_minutes").between(1, 300), 1)
        .otherwise(0)) \
    .withColumn("speed_kmh",
        when((col("trip_duration_minutes") > 0) & (col("duracao_valida") == 1),
             round(col("trip_distance_km") / (col("trip_duration_minutes") / 60), 2))
        .otherwise(None)) \
    .withColumn("speed_category",
        when(col("speed_kmh").isNull(), "duracao_invalida")
        .when(col("speed_kmh") < 10, "lenta")
        .when(col("speed_kmh") < 30, "normal")
        .otherwise("rapida"))

print("✅ Features criadas: trip_duration_minutes, duracao_valida, speed_kmh, speed_category")

display(df_featured.groupBy("speed_category").count())

In [0]:
# TAREFA 3.6: Features de Tarifa e Gorjeta
print("\n" + "="*60)
print("TAREFA 3.6: Features de Tarifa e Gorjeta")
print("="*60)

df_featured = df_featured \
    .withColumn("fare_per_km",
        when(col("trip_distance_km") > 0,
             round(col("fare_amount") / col("trip_distance_km"), 2))
        .otherwise(0)) \
    .withColumn("tip_percentage",
        when((col("fare_amount") > 0) & (col("payment_type") == 1),
             round((col("tip_amount") / col("fare_amount")) * 100, 1))
        .otherwise(None)) \
    .withColumn("tip_category",
        when(col("payment_type") != 1, "nao_aplicavel")  # dinheiro não registra gorjeta
        .when(col("tip_percentage") == 0, "sem_gorjeta")
        .when(col("tip_percentage") < 10, "baixa")
        .when(col("tip_percentage") < 20, "media")
        .otherwise("alta"))

print("✅ Features criadas: fare_per_km, tip_percentage, tip_category")
print("💡 tip_percentage só é calculado para payment_type=1 (cartão);")
print("   dinheiro recebe tip_category='nao_aplicavel' (gorjeta não registrada pelo sistema)")

display(df_featured.groupBy("tip_category").count())

In [0]:
# TAREFA 3.7: Validação Final
print("\n" + "="*60)
print("TAREFA 3.7: Validação Final")
print("="*60)

print(f"📊 Linhas finais: {df_featured.count():,}")
print(f"📋 Colunas finais: {len(df_featured.columns)}")
print(f"\n📋 Todas as colunas:")
for c in df_featured.columns:
    print(f"   - {c}")

In [0]:
# TAREFA 3.8: Salvar em Delta Lake
print("\n" + "="*60)
print("TAREFA 3.8: Salvando Dados Transformados")
print("="*60)

df_featured.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("pickup_year", "pickup_month") \
    .save(f"{PROCESSED_PATH}/featured/taxi_featured")

print(f"✅ Dados salvos em: {PROCESSED_PATH}/featured/taxi_featured")

# Validação de leitura
df_validacao = spark.read.format("delta").load(f"{PROCESSED_PATH}/featured/taxi_featured")
print(f"✅ Validação: {df_validacao.count():,} linhas lidas de volta")

In [0]:
# Diagnóstico: quanto cada regra de limpeza remove individualmente
print("=" * 60)
print("DIAGNÓSTICO: Contribuição de cada regra de limpeza")
print("=" * 60)

total = df.count()

qtd_fare_baixa = df.filter(col("fare_amount") < 2.5).count()
qtd_fare_alta = df.filter(col("fare_amount") > 500).count()
qtd_dist_alta = df.filter(col("trip_distance") > 200).count()
qtd_passenger_invalido = df.filter(~col("passenger_count").between(1, 8)).count()
qtd_tip_negativo = df.filter(col("tip_amount") < 0).count()
qtd_total_negativo = df.filter(col("total_amount") < 0).count()
qtd_dist_baixa = df.filter((col("trip_distance") > 0) & (col("trip_distance") < 0.1)).count()

print(f"fare_amount < $2.5:        {qtd_fare_baixa:,}")
print(f"fare_amount > $500:        {qtd_fare_alta:,}")
print(f"trip_distance > 200mi:     {qtd_dist_alta:,}")
print(f"passenger_count inválido:  {qtd_passenger_invalido:,}")
print(f"tip_amount < 0:            {qtd_tip_negativo:,}")
print(f"total_amount < 0:          {qtd_total_negativo:,}")
print(f"trip_distance entre 0 e 0.1mi: {qtd_dist_baixa:,}")

## 📋 Resumo — Fase 3: Feature Engineering

### Regras de Limpeza Aplicadas
Definidas com base nos percentis reais observados na Fase 2 (EDA):

| Regra | Limite Aplicado | Motivo |
|---|---|---|
| `trip_distance` | entre 0,1 e 200 milhas | Abaixo: corridas canceladas/erro de taxímetro. Acima: baseado no p99,9 real da Fase 2 |
| `fare_amount` | entre $2,5 e $500 | $2,5 é a tarifa mínima oficial em NYC. $500 baseado no p99,9 real |
| `passenger_count` | entre 1 e 8 | Fora desse intervalo é fisicamente improvável/erro de registro |
| `tip_amount` | ≥ 0 | Valores negativos são erro de sistema/estorno |
| `total_amount` | ≥ 0 | Mesma lógica acima |

### Resultado da Limpeza
- Linhas antes: 46.945.332
- Linhas depois: 46.882.150
- Removidas: 63.182 (0,13%)

### Contribuição de Cada Regra (diagnóstico individual)
| Regra violada | Registros |
|---|---|
| trip_distance entre 0 e 0.1mi | 53.888 (maior contribuinte, de longe) |
| passenger_count inválido (0 ou >8) | 6.628 |
| fare_amount < $2.5 | 2.600 |
| trip_distance > 200mi | 184 |
| fare_amount > $500 | 105 |
| total_amount < 0 | 5 |
| tip_amount < 0 | 3 |

> 💡 A soma dessas linhas (63.413) passa ligeiramente do total único removido (63.182) porque um mesmo registro pode violar mais de uma regra ao mesmo tempo — é contado 2x no diagnóstico, mas removido apenas 1x na limpeza real.

### Features Criadas (17 novas, 37 no total)
- **Conversão:** trip_distance_km (milhas → km)
- **Temporais:** pickup_hour, pickup_day_of_week, pickup_month, pickup_year,
  is_weekend, is_rush_hour, time_of_day
- **Aeroporto:** is_airport_trip, airport_type (via RatecodeID — sem cálculo geoespacial)
- **Velocidade/Duração:** trip_duration_minutes, duracao_valida, speed_kmh, speed_category
- **Tarifa/Gorjeta:** fare_per_km, tip_percentage, tip_category

### Qualidade — Duração de Viagem
- 171.592 viagens (0,37%) sinalizadas como "duracao_invalida"
  (duração fora de 1-300 minutos)
- Linhas mantidas na base (não descartadas) — só a velocidade calculada
  não é confiável para esses casos (`speed_kmh = NULL`)

### Gorjetas — Distribuição por Categoria
- alta (≥20%): 22.466.554 (48% do total) — reflete os botões pré-definidos
  de gorjeta nas máquinas de cartão de NY
- media (10-20%): 5.777.578
- baixa (<10%): 1.429.213
- sem_gorjeta: 1.078.562
- nao_aplicavel (dinheiro): 16.130.243

### Dados Salvos
- Delta Lake particionado por pickup_year, pickup_month
- Path: /Volumes/workspace/default/nyc_taxi/processed/featured/taxi_featured